# Analyze Logs — RoboMaster Lab 03

โน้ตบุ๊กนี้ใช้สำหรับอ่านไฟล์ CSV ที่ได้จากการรันหุ่นยนต์ (บันทึกโดย `src/logger.py` ลงในโฟลเดอร์ `data/raw/run_<timestamp>/`) แล้วพลอตกราฟดูผลการวิ่งแต่ละครั้ง

> รูปแบบของโน้ตบุ๊กอ้างอิงจาก `E_Kae-1.ipynb` แต่ปรับให้อ่านไฟล์จากโฟลเดอร์โปรเจกต์ในเครื่อง (`data/raw/...`) แทนการ mount Google Drive แบบ Colab และปรับชื่อ column ให้ตรงกับที่ `SensorLogger` เขียนจริง

### **สมาชิกกลุ่ม**
- ชื่อ-นามสกุล รหัสนักศึกษา
- ชื่อ-นามสกุล รหัสนักศึกษา
- ชื่อ-นามสกุล รหัสนักศึกษา


## 1. Import library ที่ต้องใช้

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


## 2. เลือก Run ที่ต้องการวิเคราะห์

`main.py` จะสร้างโฟลเดอร์ใหม่ทุกครั้งที่รัน ในรูปแบบ `data/raw/run_YYYYMMDD_HHMMSS/` แล้วเซฟไฟล์ `attitude_*.csv`, `position_*.csv`, `imu_*.csv`, `esc_*.csv` ไว้ข้างใน

ค่าเริ่มต้นด้านล่างจะเลือก **run ล่าสุด** ให้อัตโนมัติ (เรียงตามชื่อโฟลเดอร์ซึ่งเรียงตามเวลาอยู่แล้ว) ถ้าต้องการดู run อื่น ให้ uncomment บรรทัด `RUN_DIR = RAW_DIR / "run_..."` แล้วใส่ชื่อโฟลเดอร์ที่ต้องการแทน

In [ ]:
# analysis/ อยู่ใต้ root ของโปรเจกต์ 1 ชั้น -> ขึ้นไป 1 ชั้นเพื่อหา root
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PLOTS_DIR = PROJECT_ROOT / "analysis" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

run_dirs = sorted([p for p in RAW_DIR.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f"ไม่พบโฟลเดอร์ run ใดๆ ใน {RAW_DIR} -- ลองรัน main.py เพื่อเก็บข้อมูลก่อน")

RUN_DIR = run_dirs[-1]  # เลือก run ล่าสุด
# RUN_DIR = RAW_DIR / "run_20260714_153000"  # <-- หรือระบุ run ที่ต้องการเองตรงนี้

print("Run ที่เลือก:", RUN_DIR.name)
print("Path เต็ม:", RUN_DIR)


## 3. หา path ของ CSV แต่ละประเภทอัตโนมัติ

In [ ]:
candidate_paths = [str(p) for p in RUN_DIR.glob("*.csv")]

csv_files = {}
for name in ["attitude", "position", "imu", "esc"]:
    matches = [p for p in candidate_paths if name in Path(p).name.lower()]
    csv_files[name] = matches[0] if matches else None

print("พบไฟล์ดังนี้:")
for name, path in csv_files.items():
    print(f"  {name}: {path}")


## 4. ตั้งค่าชื่อ column เวลา และหน่วยของแต่ละค่า

`SensorLogger` (ใน `src/logger.py`) เขียน column เวลาเป็น `elapsed_s` (จำนวนวินาทีนับจากเริ่ม log ของ run นั้นๆ อยู่แล้ว ไม่ใช่ unix timestamp มิลลิวินาทีเหมือนตัวอย่างเดิม) และตั้งชื่อ column แบบกำกับหน่วยไว้ในชื่อ column เลย (เช่น `yaw_deg`, `x_m`, `speed_1_rpm`) ซึ่งตรงกับหน่วยที่ RoboMaster SDK ระบุไว้ในเอกสาร `sub_attitude`, `sub_position`, `sub_imu`, `sub_esc` ยกเว้นค่า `acc_*` และ `gyro_*` จาก `sub_imu` ที่เอกสารทางการไม่ได้ระบุหน่วยไว้ชัดเจน จึงกำกับตามธรรมเนียมทั่วไปของ IMU (m/s² และ deg/s) และมาร์กว่าเป็นค่าประมาณ (approx.)

ข้อควรระวัง: `z_deg` ใน position **ไม่ใช่ระยะทาง** แต่เป็นมุมหมุนรอบแกน z จึงแยกไปพลอตอีกกราฟหนึ่งจาก `x_m`, `y_m`

In [ ]:
TIME_COL = "elapsed_s"       # ชื่อ column เวลา (วินาที นับจากเริ่ม log ของ run)
TIME_IS_MILLISECONDS = False # elapsed_s เป็นวินาทีอยู่แล้ว ไม่ต้องหาร 1000

# ชื่อที่แสดงผล + หน่วย ของแต่ละ column
# อ้างอิงชื่อ column ตรงจาก src/logger.py (SensorLogger) และหน่วยจาก RoboMaster Developer Guide
COLUMN_INFO = {
    "yaw_deg":          ("Yaw", "deg"),
    "pitch_deg":        ("Pitch", "deg"),
    "roll_deg":         ("Roll", "deg"),
    "x_m":              ("X Position", "m"),
    "y_m":              ("Y Position", "m"),
    "z_deg":            ("Z Rotation", "deg"),        # ไม่ใช่ระยะทาง เป็นมุมหมุนรอบแกน z
    "acc_x_sdk_unit":   ("Acc X", "m/s^2 (approx.)"),
    "acc_y_sdk_unit":   ("Acc Y", "m/s^2 (approx.)"),
    "acc_z_sdk_unit":   ("Acc Z", "m/s^2 (approx.)"),
    "gyro_x_sdk_unit":  ("Gyro X", "deg/s (approx.)"),
    "gyro_y_sdk_unit":  ("Gyro Y", "deg/s (approx.)"),
    "gyro_z_sdk_unit":  ("Gyro Z", "deg/s (approx.)"),
    "speed_1_rpm":      ("Wheel 1 Speed", "rpm"),
    "speed_2_rpm":      ("Wheel 2 Speed", "rpm"),
    "speed_3_rpm":      ("Wheel 3 Speed", "rpm"),
    "speed_4_rpm":      ("Wheel 4 Speed", "rpm"),
    "angle_1_raw":      ("Wheel 1 Angle", "deg (raw code)"),
    "angle_2_raw":      ("Wheel 2 Angle", "deg (raw code)"),
    "angle_3_raw":      ("Wheel 3 Angle", "deg (raw code)"),
    "angle_4_raw":      ("Wheel 4 Angle", "deg (raw code)"),
}


## 5. ฟังก์ชันสำหรับ plot กราฟ

In [ ]:
def add_axis_arrowheads(ax):
    """ใส่หัวลูกศรที่ปลายแกน x (ขวา) และแกน y (บน) แบบกราฟฟิสิกส์/วิศวกรรม"""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.plot(1, 0, ">k", markersize=9, transform=ax.transAxes, clip_on=False)
    ax.plot(0, 1, "^k", markersize=9, transform=ax.transAxes, clip_on=False)


def plot_csv(csv_path, title, y_columns, save_png=True):
    if csv_path is None:
        print(f"[WARN] ไม่พบไฟล์สำหรับ: {title}")
        return None

    df = pd.read_csv(csv_path)

    if TIME_COL not in df.columns:
        raise KeyError(
            f"ไม่พบ column เวลา '{TIME_COL}' ใน {csv_path}. "
            f"column ที่มีอยู่คือ: {list(df.columns)} -- แก้ค่า TIME_COL ในเซลล์ด้านบนให้ตรงกับชื่อจริง"
        )

    time_values = df[TIME_COL]
    if TIME_IS_MILLISECONDS:
        time_values = time_values / 1000.0

    fig, ax = plt.subplots(figsize=(12, 6))

    units_used = set()
    for col in y_columns:
        if col not in df.columns:
            continue
        label, unit = COLUMN_INFO.get(col, (col, ""))
        units_used.add(unit)
        legend_label = f"{label} ({unit})" if unit else label
        ax.plot(time_values, df[col], label=legend_label)

    if len(units_used) == 1:
        y_unit = units_used.pop()
        y_label = f"Value ({y_unit})" if y_unit else "Value"
    else:
        y_label = "Value (ดูหน่วยที่ legend)"

    ax.set_title(title)
    ax.set_xlabel("Elapsed Time (s)")
    ax.set_ylabel(y_label)
    ax.legend()
    ax.grid(True)

    add_axis_arrowheads(ax)

    fig.tight_layout()

    if save_png:
        png_path = PLOTS_DIR / f"{RUN_DIR.name}_{Path(csv_path).stem}.png"
        fig.savefig(png_path, dpi=200)
        print(f"[SAVE] {png_path}")

    plt.show()
    return df


## 6. กราฟ Attitude (yaw, pitch, roll)

In [ ]:
df_attitude = plot_csv(
    csv_files["attitude"],
    "RoboMaster Attitude vs Elapsed Time",
    ["yaw_deg", "pitch_deg", "roll_deg"],
)


กราฟนี้แสดงมุมเอียงของตัวหุ่นยนต์ทั้งสามแกน yaw คือการหันซ้าย-ขวา pitch คือการก้ม-เงย และ roll คือการเอียงซ้าย-ขวา ทั้งสามค่าหน่วยเป็นองศาเหมือนกันเลยเอามาโชว์ในกราฟเดียวกันได้ ถ้าเส้นไหนแกว่งแรงหรือมีสัญญาณ spike แปลว่าช่วงนั้นหุ่นยนต์น่าจะโดนกระแทกหรือเลี้ยวกะทันหัน ส่วนช่วงที่เส้นเรียบนิ่งคือช่วงที่หุ่นยนต์วิ่งตรงหรือหยุดอยู่กับที่

## 7. กราฟ Position — ระยะทาง (x, y)

In [ ]:
df_position = plot_csv(
    csv_files["position"],
    "RoboMaster Position (X, Y) vs Elapsed Time",
    ["x_m", "y_m"],
)


กราฟนี้ตัดเอาเฉพาะ x กับ y ที่เป็นระยะทางจริง หน่วยเมตร มาโชว์ด้วยกัน เพราะ `z_deg` ในไฟล์นี้ไม่ใช่ระยะทางในแกน z แต่เป็นมุมหมุน (ตรวจสอบจากเอกสาร SDK แล้วพบว่า `sub_position` คืนค่า z เป็นองศา ไม่ใช่เมตร) ถ้าเอามารวมกันจะทำให้อ่านกราฟผิด เลยแยกออกไปอีกกราฟหนึ่ง เส้น x กับ y ที่ขึ้นๆ ลงๆ บอกได้คร่าวๆ ว่าหุ่นยนต์เคลื่อนที่ไปทิศไหนบ้างเมื่อเทียบกับจุดเริ่มต้น

## 8. กราฟ Position — มุมหมุน (z)

In [ ]:
_ = plot_csv(
    csv_files["position"],
    "RoboMaster Position Z-Rotation vs Elapsed Time",
    ["z_deg"],
    save_png=False,
)


แยกกราฟนี้ออกมาต่างหากเพราะ `z_deg` หน่วยเป็นองศา ไม่ใช่เมตรเหมือน `x_m`, `y_m` ถ้าดูแล้วเส้นเพิ่มขึ้นเรื่อยๆ แปลว่าหุ่นยนต์หมุนไปทางเดียวต่อเนื่อง แต่ถ้าขึ้นๆ ลงๆ สลับกันแปลว่ามีการหมุนกลับไปกลับมา

## 9. กราฟ IMU — Acceleration (acc_x, acc_y, acc_z)

In [ ]:
df_imu = plot_csv(
    csv_files["imu"],
    "RoboMaster IMU - Acceleration vs Elapsed Time",
    ["acc_x_sdk_unit", "acc_y_sdk_unit", "acc_z_sdk_unit"],
)


ค่าอัตราเร่งทั้งสามแกนจากตัว IMU ของหุ่นยนต์ เอกสาร SDK ไม่ได้บอกหน่วยตรงๆ ของค่านี้ ในกราฟจึงกำกับเป็น m/s² (approx.) ตามมาตรฐานทั่วไปของเซนเซอร์ accelerometer ไม่ได้ยืนยันจากผู้ผลิตโดยตรง ช่วงที่กราฟมี peak แหลมๆ มักจะตรงกับตอนที่หุ่นยนต์ออกตัว เบรกกะทันหัน หรือโดนกระแทก

## 10. กราฟ IMU — Gyro (gyro_x, gyro_y, gyro_z)

In [ ]:
_ = plot_csv(
    csv_files["imu"],
    "RoboMaster IMU - Gyro vs Elapsed Time",
    ["gyro_x_sdk_unit", "gyro_y_sdk_unit", "gyro_z_sdk_unit"],
    save_png=False,
)


ค่าความเร็วเชิงมุม (angular rate) ทั้งสามแกน กำกับหน่วยเป็น deg/s (approx.) ด้วยเหตุผลเดียวกับ acceleration คือ SDK ไม่ได้ระบุหน่วยไว้ชัดเจน แยกกราฟออกจาก acceleration เพราะเป็นคนละปริมาณและคนละหน่วยกัน เส้นที่นิ่งใกล้ศูนย์แปลว่าหุ่นยนต์ไม่ได้หมุนตัว ส่วนช่วงที่ค่าพุ่งขึ้นแรงคือช่วงที่หุ่นยนต์หมุนตัวเร็ว

## 11. กราฟ ESC — ความเร็วล้อ (speed_1..4)

In [ ]:
df_esc = plot_csv(
    csv_files["esc"],
    "RoboMaster ESC Speed vs Elapsed Time",
    ["speed_1_rpm", "speed_2_rpm", "speed_3_rpm", "speed_4_rpm"],
)


ความเร็วของมอเตอร์ล้อทั้ง 4 ล้อ หน่วย rpm ตามที่ระบุไว้ใน `sub_esc` ของเอกสาร SDK ถ้าเส้นทั้ง 4 เส้นวิ่งขนานกันแปลว่าหุ่นยนต์วิ่งตรง แต่ถ้าเส้นไหนแยกออกจากเส้นอื่นชัดๆ แปลว่าน่าจะมีการเลี้ยวหรือหมุนตัวอยู่ในช่วงนั้น

## 12. กราฟ ESC — มุมล้อ (angle_1..4)

In [ ]:
_ = plot_csv(
    csv_files["esc"],
    "RoboMaster ESC Angle vs Elapsed Time",
    ["angle_1_raw", "angle_2_raw", "angle_3_raw", "angle_4_raw"],
    save_png=False,
)


มุมของมอเตอร์แต่ละล้อ ค่าดิบจาก encoder อยู่ในช่วง 0-32767 ซึ่งเอกสาร SDK ระบุว่าแมพมาจาก 0-360 องศา ในกราฟนี้เลยกำกับหน่วยเป็น deg (raw code) เพื่อให้รู้ว่าเป็นค่าที่แปลงมาจาก encoder ไม่ใช่ค่ามุมที่วัดตรงๆ กราฟที่เป็นฟันปลาซ้ำๆ กันเป็นเรื่องปกติ เพราะล้อหมุนครบรอบก็จะรีเซ็ตมุมกลับไปที่ 0 ใหม่